# 05 — Heavy-M: ResNet-18 Multi-Scale Saliency Training

This notebook trains the **Heavy-M** reference model:

- **encoder:** ImageNet-pretrained ResNet-18;
- **decoder:** the same compact multi-scale top-down decoder used by Light-M;
- **feature levels:** `s4`, `s8`, `s16`, and `s32`;
- **training objective:** `KLD + 0.5 × (1 − CC)`;
- **checkpoint rule:** highest validation CC.

Heavy-M is the conventional higher-capacity reference in the controlled experiment. Comparing it with
Light-M isolates the effect of encoder capacity while keeping the decoder, data, loss, optimizer policy,
validation protocol, and output resolution fixed.

The notebook is divided into small, readable cells, with a Markdown explanation before every code cell.
All important artifacts are written directly to Google Drive.


## What this notebook saves

The complete Heavy-M run produces:

- `checkpoints/heavy_multi/best.pt`;
- `checkpoints/heavy_multi/last.pt`;
- `outputs/metrics/heavy_multi_history.csv`;
- `outputs/metrics/heavy_multi_validation_metrics.csv`;
- `outputs/metrics/heavy_multi_validation_per_image.csv`;
- `outputs/metrics/heavy_multi_parameter_summary.csv`;
- `outputs/figures/training_curves_heavy_multi.png`;
- `outputs/figures/heavy_multi_validation_preview.png`;
- `logs/run_notes/heavy_multi_config.json`;
- `logs/run_notes/heavy_multi_run_summary.json`.

The final internal test split is intentionally **not** evaluated here. It remains untouched for the final
comparison notebook.


## 1. Mount Google Drive

Google Drive is the persistent storage layer. The dataset source, persistent TAR archive, checkpoints,
histories, figures, and run metadata are all stored there.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Define the experiment configuration

This is the only cell that should normally be edited before a run. The stable model name controls all
output paths, while the data, optimizer, and validation settings remain consistent with the other models.

In [ ]:
from pathlib import Path
import os

# ------------------------------------------------------------------
# Repository and persistent project roots
# ------------------------------------------------------------------
REPO_URL = "https://github.com/RicoDalB/efficient-saliency-prediction.git"
REPO_ROOT = Path("/content/efficient-saliency-prediction")

DRIVE_ROOT = Path("/content/drive/MyDrive/saliency_project")
DRIVE_DATA_ROOT = DRIVE_ROOT / "data" / "SALICON"

CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
LOG_ROOT = DRIVE_ROOT / "logs"

# ------------------------------------------------------------------
# Model identity
# ------------------------------------------------------------------
MODEL_NAME = "heavy_multi"
MODEL_LABEL = "Heavy-M"
DECODER_CHANNELS = 32
PRETRAINED = True

# ------------------------------------------------------------------
# Reproducible data and training settings
# ------------------------------------------------------------------
SEED = 42
OUTPUT_SIZE = (192, 256)  # (height, width)
BATCH_SIZE = 8  # Use 4 only if the allocated Colab GPU reports CUDA OOM.
NUM_WORKERS = min(4, os.cpu_count() or 2)
PREFETCH_FACTOR = 2

MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
MIN_CC_IMPROVEMENT = 1e-4

ENCODER_LR = 1e-4
DECODER_LR = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
FREEZE_ENCODER_EPOCHS = 1
USE_AMP = True

# Resume from last.pt after an interrupted runtime.
RESUME_TRAINING = True

# Safety guard. Keep False unless an existing completed run should be replaced.
ALLOW_OVERWRITE_EXISTING_RUN = False

print("Model:", MODEL_NAME)
print("Output size:", OUTPUT_SIZE)
print("Batch size:", BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)

## 3. Clone or update the GitHub repository

The notebook uses the reusable code in `src/`. A fresh runtime clones the repository; an existing local
clone is updated with a fast-forward-only pull.

In [ ]:
import subprocess

if (REPO_ROOT / ".git").is_dir():
    print("Repository already exists. Pulling the latest main branch...")
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
        check=True,
    )
else:
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

commit_hash = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_ROOT)
print("Git commit:", commit_hash)

## 3.1 Verify the Heavy-M source contract

Heavy-M requires both the ResNet-18 encoder and the shared multi-scale decoder. This cell checks the temporary
GitHub clone for two legacy spelling mistakes that would prevent Heavy-M from running:

- `farward` instead of `forward` in `ResNet18Encoder`;
- `_unsample_add` instead of `_upsample_add` in `MultiScaleDecoder`.

When either spelling is found, only the temporary clone under `/content` is corrected. The notebook records
the correction in the run configuration. The persistent GitHub repository is not modified by this cell.


In [ ]:
MODELS_SOURCE_PATH = REPO_ROOT / "src" / "models.py"
source_patch_notes = []

if not MODELS_SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Missing model source file: {MODELS_SOURCE_PATH}")

models_source = MODELS_SOURCE_PATH.read_text(encoding="utf-8")

known_source_fixes = {
    "def farward(self, x: Tensor)": "def forward(self, x: Tensor)",
    "def _unsample_add(self, x: Tensor, lateral: Tensor)": (
        "def _upsample_add(self, x: Tensor, lateral: Tensor)"
    ),
}

for incorrect_text, corrected_text in known_source_fixes.items():
    if incorrect_text in models_source:
        models_source = models_source.replace(incorrect_text, corrected_text)
        source_patch_notes.append(f"{incorrect_text} -> {corrected_text}")

if source_patch_notes:
    MODELS_SOURCE_PATH.write_text(models_source, encoding="utf-8")
    print("Applied temporary compatibility corrections:")
    for note in source_patch_notes:
        print(" -", note)
else:
    print("The Heavy-M source contract is already correct.")

verified_source = MODELS_SOURCE_PATH.read_text(encoding="utf-8")

if "class ResNet18Encoder" not in verified_source:
    raise RuntimeError("ResNet18Encoder is missing from src/models.py.")

if 'if name == "heavy_multi"' not in verified_source:
    raise RuntimeError("The heavy_multi factory branch is missing from src/models.py.")

if "def forward(self, x: Tensor)" not in verified_source:
    raise RuntimeError("A valid encoder forward method could not be verified.")

if "def _upsample_add(self, x: Tensor, lateral: Tensor)" not in verified_source:
    raise RuntimeError("The multi-scale decoder helper could not be verified.")


## 4. Import the shared project modules

No notebook-level copy of the model, loss, or training loop is created. The experiment imports the stable
implementation from `src/`, which keeps the three neural-model runs comparable.

In [ ]:
import sys
import importlib

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.data as data_module
import src.losses_metrics as losses_module
import src.models as models_module
import src.train_eval as train_eval_module

# Reload explicitly so a pulled source update is visible without autoreload.
importlib.reload(data_module)
importlib.reload(losses_module)
importlib.reload(models_module)
importlib.reload(train_eval_module)

from src.data import SaliconDataset
from src.losses_metrics import saliency_loss, spatial_softmax
from src.models import build_model, parameter_summary
from src.train_eval import (
    build_optimizer,
    evaluate_and_save,
    fit_model,
    load_checkpoint,
    plot_training_history,
)

print("Shared modules imported successfully.")

## 5. Record the runtime and select the device

Reproducibility requires fixed random seeds and a clear description of the runtime used for training.
Automatic mixed precision is enabled only when CUDA is available.

In [ ]:
import json
import platform
import random

import numpy as np
import pandas as pd
import torch
import torchvision

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

print("Device:", DEVICE)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB",
    )

## 6. Create all persistent output folders

These directories live in Google Drive. The notebook may lose everything under `/content`, but these
outputs survive runtime termination.

In [ ]:
MODEL_CHECKPOINT_DIR = CHECKPOINT_ROOT / MODEL_NAME

HISTORY_CSV_PATH = METRICS_ROOT / f"{MODEL_NAME}_history.csv"
VALIDATION_SUMMARY_PATH = METRICS_ROOT / f"{MODEL_NAME}_validation_metrics.csv"
VALIDATION_PER_IMAGE_PATH = METRICS_ROOT / f"{MODEL_NAME}_validation_per_image.csv"
PARAMETER_SUMMARY_PATH = METRICS_ROOT / f"{MODEL_NAME}_parameter_summary.csv"

CURVES_FIGURE_PATH = FIGURE_ROOT / f"training_curves_{MODEL_NAME}.png"
PREVIEW_FIGURE_PATH = FIGURE_ROOT / f"{MODEL_NAME}_validation_preview.png"

RUN_NOTE_ROOT = LOG_ROOT / "run_notes"
CONFIG_JSON_PATH = RUN_NOTE_ROOT / f"{MODEL_NAME}_config.json"
RUN_SUMMARY_JSON_PATH = RUN_NOTE_ROOT / f"{MODEL_NAME}_run_summary.json"

for directory in [
    MODEL_CHECKPOINT_DIR,
    METRICS_ROOT,
    FIGURE_ROOT,
    RUN_NOTE_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = MODEL_CHECKPOINT_DIR / "best.pt"
LAST_CHECKPOINT_PATH = MODEL_CHECKPOINT_DIR / "last.pt"

print("Checkpoint directory:", MODEL_CHECKPOINT_DIR)
print("Metrics directory:", METRICS_ROOT)
print("Figure directory:", FIGURE_ROOT)

# Part A — Fast local SALICON cache

Training must not repeatedly open thousands of small files through mounted Drive. The workflow therefore
uses:

- Drive as the permanent source of truth;
- one persistent TAR archive in Drive;
- an extracted temporary cache under `/content` for fast batch loading.

On later runtimes, the notebook copies and extracts the existing TAR. The slow file-by-file copy is used
only if the persistent archive does not yet exist.

## 7. Define cache paths and disk-space helpers

The marker file is created only after the local image and map folders have been verified.

In [ ]:
import shutil
import time

SALICON_ARCHIVE_DRIVE = DRIVE_ROOT / "backups" / "salicon_images_maps.tar"
TEMPORARY_DRIVE_ARCHIVE = Path(str(SALICON_ARCHIVE_DRIVE) + ".tmp")

LOCAL_CACHE_ROOT = Path("/content/saliency_cache")
LOCAL_DATA_ROOT = LOCAL_CACHE_ROOT / "SALICON"
LOCAL_ARCHIVE_PATH = Path("/content/salicon_images_maps.tar")
CACHE_READY_MARKER = LOCAL_DATA_ROOT / ".cache_ready"

DRIVE_IMAGE_ROOT = DRIVE_DATA_ROOT / "images"
DRIVE_MAP_ROOT = DRIVE_DATA_ROOT / "maps"

LOCAL_IMAGE_ROOT = LOCAL_DATA_ROOT / "images"
LOCAL_MAP_ROOT = LOCAL_DATA_ROOT / "maps"

SALICON_ARCHIVE_DRIVE.parent.mkdir(parents=True, exist_ok=True)


def print_local_disk_space() -> None:
    total, used, free = shutil.disk_usage("/content")

    print(f"Local disk total: {total / (1024 ** 3):.2f} GB")
    print(f"Local disk used:  {used / (1024 ** 3):.2f} GB")
    print(f"Local disk free:  {free / (1024 ** 3):.2f} GB")


def local_cache_is_valid() -> bool:
    return (
        CACHE_READY_MARKER.is_file()
        and LOCAL_IMAGE_ROOT.is_dir()
        and LOCAL_MAP_ROOT.is_dir()
    )


print_local_disk_space()
print("Persistent archive:", SALICON_ARCHIVE_DRIVE)
print("Local cache:", LOCAL_DATA_ROOT)

## 8. Define resumable file-copy helpers

This fallback path is required only for the one-time archive creation. Existing destination files with the
same byte size are reused, and transient Drive errors are retried.

In [ ]:
def list_regular_files(root: Path) -> list[Path]:
    if not root.is_dir():
        raise FileNotFoundError(f"Required source folder does not exist: {root}")

    files = sorted(path for path in root.rglob("*") if path.is_file())

    if not files:
        raise RuntimeError(f"No files were found under: {root}")

    return files


def copy_file_with_retries(
    source: Path,
    destination: Path,
    *,
    maximum_attempts: int = 3,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None

    for attempt in range(1, maximum_attempts + 1):
        try:
            shutil.copy2(source, destination)
            return
        except OSError as error:
            last_error = error

            if attempt < maximum_attempts:
                wait_seconds = attempt * 2
                print(
                    f"\nTemporary copy error for {source.name}. "
                    f"Retrying in {wait_seconds} seconds..."
                )
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"Could not copy {source} after {maximum_attempts} attempts."
    ) from last_error


def copy_tree_to_local_cache(
    source_root: Path,
    destination_root: Path,
    *,
    label: str,
    progress_interval: int = 250,
) -> tuple[int, int]:
    source_files = list_regular_files(source_root)

    total_files = len(source_files)
    total_bytes = sum(path.stat().st_size for path in source_files)

    destination_root.mkdir(parents=True, exist_ok=True)

    print(
        f"\n{label}: {total_files:,} files, "
        f"{total_bytes / (1024 ** 3):.2f} GB"
    )

    copied_files = 0
    reused_files = 0
    started_at = time.perf_counter()

    for index, source_path in enumerate(source_files, start=1):
        relative_path = source_path.relative_to(source_root)
        destination_path = destination_root / relative_path
        source_size = source_path.stat().st_size

        if (
            destination_path.is_file()
            and destination_path.stat().st_size == source_size
        ):
            reused_files += 1
        else:
            copy_file_with_retries(source_path, destination_path)
            copied_files += 1

        if (
            index == 1
            or index % progress_interval == 0
            or index == total_files
        ):
            elapsed = time.perf_counter() - started_at
            rate = index / max(elapsed, 1e-6)

            print(
                f"\r{label}: {index:,}/{total_files:,} files processed | "
                f"copied {copied_files:,} | reused {reused_files:,} | "
                f"{rate:.1f} files/s",
                end="",
                flush=True,
            )

    print()

    missing_or_incomplete = []

    for source_path in source_files:
        relative_path = source_path.relative_to(source_root)
        destination_path = destination_root / relative_path

        if (
            not destination_path.is_file()
            or destination_path.stat().st_size != source_path.stat().st_size
        ):
            missing_or_incomplete.append(relative_path.as_posix())

            if len(missing_or_incomplete) >= 10:
                break

    if missing_or_incomplete:
        raise RuntimeError(
            f"{label} local copy verification failed. "
            f"Examples: {missing_or_incomplete}"
        )

    print(f"{label} local copy verified successfully.")

    return total_files, total_bytes

## 9. Define persistent archive creation and extraction

Archive creation reads only from the verified local cache and writes one sequential file to Drive. A
temporary suffix prevents an interrupted archive write from being mistaken for a valid backup.

In [ ]:
def create_persistent_archive_from_local_cache() -> None:
    if TEMPORARY_DRIVE_ARCHIVE.exists():
        TEMPORARY_DRIVE_ARCHIVE.unlink()

    print("\nCreating the persistent TAR from the verified local cache...")
    print("The source is local; Drive receives only one sequential file.")

    command = [
        "tar",
        "-cf",
        str(TEMPORARY_DRIVE_ARCHIVE),
        "-C",
        str(LOCAL_CACHE_ROOT),
        "SALICON/images",
        "SALICON/maps",
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        TEMPORARY_DRIVE_ARCHIVE.unlink(missing_ok=True)
        raise RuntimeError(
            "Persistent TAR creation failed.\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    if (
        not TEMPORARY_DRIVE_ARCHIVE.is_file()
        or TEMPORARY_DRIVE_ARCHIVE.stat().st_size == 0
    ):
        raise RuntimeError("The temporary persistent archive is missing or empty.")

    SALICON_ARCHIVE_DRIVE.unlink(missing_ok=True)
    TEMPORARY_DRIVE_ARCHIVE.replace(SALICON_ARCHIVE_DRIVE)

    print(
        "Persistent archive saved successfully:",
        f"{SALICON_ARCHIVE_DRIVE.stat().st_size / (1024 ** 3):.2f} GB",
    )
    print(SALICON_ARCHIVE_DRIVE)


def extract_existing_persistent_archive() -> None:
    archive_size = SALICON_ARCHIVE_DRIVE.stat().st_size
    required_free_bytes = archive_size * 2 + 2 * (1024 ** 3)
    free_bytes = shutil.disk_usage("/content").free

    if free_bytes < required_free_bytes:
        raise RuntimeError(
            "Insufficient local disk space for archive copy and extraction. "
            f"Free: {free_bytes / (1024 ** 3):.2f} GB; "
            f"recommended minimum: {required_free_bytes / (1024 ** 3):.2f} GB."
        )

    print("\nPersistent SALICON archive found in Drive.")
    print(f"Archive size: {archive_size / (1024 ** 3):.2f} GB")
    print("Copying the single archive to local disk...")

    LOCAL_ARCHIVE_PATH.unlink(missing_ok=True)
    shutil.copy2(SALICON_ARCHIVE_DRIVE, LOCAL_ARCHIVE_PATH)

    if LOCAL_CACHE_ROOT.exists():
        shutil.rmtree(LOCAL_CACHE_ROOT)

    LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

    print("Extracting SALICON into the local cache...")

    result = subprocess.run(
        [
            "tar",
            "-xf",
            str(LOCAL_ARCHIVE_PATH),
            "-C",
            str(LOCAL_CACHE_ROOT),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "SALICON extraction failed.\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    if not LOCAL_IMAGE_ROOT.is_dir():
        raise RuntimeError("The extracted image folder was not found.")

    if not LOCAL_MAP_ROOT.is_dir():
        raise RuntimeError("The extracted map folder was not found.")

    CACHE_READY_MARKER.touch()
    LOCAL_ARCHIVE_PATH.unlink(missing_ok=True)

    print("Local SALICON cache extracted and verified successfully.")

## 10. Prepare or reuse the local cache

Normal future runs enter the second branch: copy the existing TAR from Drive, extract it, verify it, and
remove the temporary local TAR. The one-time file-by-file fallback is kept so the notebook remains fully
executable from a clean project state.

In [ ]:
print_local_disk_space()

# Remove a partial local TAR left by an interrupted runtime.
LOCAL_ARCHIVE_PATH.unlink(missing_ok=True)

if local_cache_is_valid():
    print("\nLocal SALICON cache is already ready.")

elif SALICON_ARCHIVE_DRIVE.is_file():
    extract_existing_persistent_archive()

else:
    print("\nNo persistent archive was found.")
    print("Preparing the one-time local cache directly from Drive...")
    print(
        "This reads every image and map once, prints progress, "
        "and reuses completed files."
    )

    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    CACHE_READY_MARKER.unlink(missing_ok=True)

    image_file_count, image_bytes = copy_tree_to_local_cache(
        DRIVE_IMAGE_ROOT,
        LOCAL_IMAGE_ROOT,
        label="Images",
    )

    map_file_count, map_bytes = copy_tree_to_local_cache(
        DRIVE_MAP_ROOT,
        LOCAL_MAP_ROOT,
        label="Maps",
    )

    total_dataset_bytes = image_bytes + map_bytes

    print(
        f"\nVerified local dataset: "
        f"{image_file_count + map_file_count:,} files, "
        f"{total_dataset_bytes / (1024 ** 3):.2f} GB"
    )

    CACHE_READY_MARKER.touch()
    create_persistent_archive_from_local_cache()

if not local_cache_is_valid():
    raise RuntimeError("The local SALICON cache did not pass final validation.")

TRAINING_DATA_ROOT = LOCAL_DATA_ROOT

print("\nTraining data root:", TRAINING_DATA_ROOT)
print_local_disk_space()

## 11. Verify the extracted dataset structure

This fast check confirms that the expected image and map trees exist before manifest processing begins.
It does not reopen every file.

In [ ]:
image_count = sum(1 for path in LOCAL_IMAGE_ROOT.rglob("*") if path.is_file())
map_count = sum(1 for path in LOCAL_MAP_ROOT.rglob("*") if path.is_file())

print(f"Local image files: {image_count:,}")
print(f"Local map files:   {map_count:,}")
print(f"Total local files: {image_count + map_count:,}")

if image_count == 0 or map_count == 0:
    raise RuntimeError("The local cache contains no usable images or maps.")

# Part B — Local manifests and fast DataLoaders

The version-controlled split manifests remain unchanged. Temporary copies under `/content` contain paths
relative to the extracted SALICON root, so the Dataset reads only from the local cache.

## 12. Localize and validate the frozen manifests

The function supports already-relative paths and older absolute Drive paths. Every referenced image and map
is checked before the long training run begins.

In [ ]:
LOCAL_MANIFEST_ROOT = Path("/content/saliency_manifests") / MODEL_NAME
LOCAL_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_MANIFESTS = {
    "train": REPO_ROOT / "splits" / "train_manifest.csv",
    "val": REPO_ROOT / "splits" / "val_manifest.csv",
    "test": REPO_ROOT / "splits" / "test_manifest.csv",
}

IMAGE_COLUMN = "image_relpath"
MAP_COLUMN = "map_relpath"
ID_COLUMN = "sample_id"


def salicon_relative_path(path_value: object) -> str:
    text = str(path_value).replace("\\", "/")

    for marker in ("/SALICON/", "data/SALICON/", "SALICON/"):
        if marker in text:
            text = text.split(marker, 1)[1]
            break

    relative_path = Path(text)

    if relative_path.is_absolute():
        raise ValueError(
            "Could not convert an absolute manifest path into a SALICON-relative path: "
            f"{path_value}"
        )

    return relative_path.as_posix()


def localize_manifest(split_name: str, source_path: Path) -> Path:
    if not source_path.is_file():
        raise FileNotFoundError(f"Missing source manifest: {source_path}")

    dataframe = pd.read_csv(source_path)

    required_columns = {IMAGE_COLUMN, MAP_COLUMN, ID_COLUMN}
    missing_columns = required_columns.difference(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name} manifest is missing columns: {sorted(missing_columns)}"
        )

    localized = dataframe.copy()
    localized[IMAGE_COLUMN] = localized[IMAGE_COLUMN].map(salicon_relative_path)
    localized[MAP_COLUMN] = localized[MAP_COLUMN].map(salicon_relative_path)

    missing_examples = []

    for row in localized.itertuples(index=False):
        image_path = TRAINING_DATA_ROOT / getattr(row, IMAGE_COLUMN)
        map_path = TRAINING_DATA_ROOT / getattr(row, MAP_COLUMN)

        if not image_path.is_file() or not map_path.is_file():
            missing_examples.append(
                {
                    "sample_id": str(getattr(row, ID_COLUMN)),
                    "image": str(image_path),
                    "map": str(map_path),
                }
            )

            if len(missing_examples) >= 10:
                break

    if missing_examples:
        raise RuntimeError(
            f"{split_name} manifest does not resolve inside the local cache. "
            f"Examples: {missing_examples}"
        )

    output_path = LOCAL_MANIFEST_ROOT / f"{split_name}_manifest.csv"
    localized.to_csv(output_path, index=False)

    print(f"{split_name:>5}: {len(localized):,} samples -> {output_path}")
    return output_path


LOCAL_MANIFESTS = {
    split_name: localize_manifest(split_name, source_path)
    for split_name, source_path in SOURCE_MANIFESTS.items()
}

## 13. Create the training and validation datasets

Both splits use the same resize, ImageNet normalization, and target-distribution normalization. Only the
sample order differs.

In [ ]:
train_dataset = SaliconDataset(
    manifest_path=LOCAL_MANIFESTS["train"],
    data_root=TRAINING_DATA_ROOT,
    image_column=IMAGE_COLUMN,
    map_column=MAP_COLUMN,
    id_column=ID_COLUMN,
    output_size=OUTPUT_SIZE,
    use_imagenet_normalization=True,
)

val_dataset = SaliconDataset(
    manifest_path=LOCAL_MANIFESTS["val"],
    data_root=TRAINING_DATA_ROOT,
    image_column=IMAGE_COLUMN,
    map_column=MAP_COLUMN,
    id_column=ID_COLUMN,
    output_size=OUTPUT_SIZE,
    use_imagenet_normalization=True,
)

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")

## 14. Build tuned local DataLoaders

Persistent workers and prefetching are enabled only when multiprocessing is active. The same seed controls
the shuffled training order.

In [ ]:
from torch.utils.data import DataLoader


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

common_loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "worker_init_fn": seed_worker,
}

if NUM_WORKERS > 0:
    common_loader_kwargs.update(
        {
            "persistent_workers": True,
            "prefetch_factor": PREFETCH_FACTOR,
        }
    )

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=loader_generator,
    drop_last=False,
    **common_loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **common_loader_kwargs,
)

print(f"Training batches:   {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

## 15. Measure DataLoader speed

The first batch includes worker startup. The following warm batches better represent the loading speed seen
during training.

In [ ]:
def benchmark_dataloader(
    dataloader: DataLoader,
    *,
    number_of_batches: int = 20,
) -> dict[str, float]:
    iterator = iter(dataloader)
    measured = min(number_of_batches, len(dataloader))

    batch_times = []

    for _ in range(measured):
        start = time.perf_counter()
        next(iterator)
        batch_times.append(time.perf_counter() - start)

    return {
        "measured_batches": measured,
        "first_batch_seconds": batch_times[0],
        "warm_batch_mean_seconds": (
            float(np.mean(batch_times[1:]))
            if len(batch_times) > 1
            else batch_times[0]
        ),
        "total_seconds": float(sum(batch_times)),
    }


loader_benchmark = benchmark_dataloader(train_loader)

print(loader_benchmark)

## 16. Inspect one real batch

The model contract requires RGB tensors `[B, 3, H, W]`, saliency targets `[B, 1, H, W]`, and target mass
equal to one for every image.

In [ ]:
sample_batch = next(iter(train_loader))

batch_images = sample_batch["image"]
batch_targets = sample_batch["target"]
batch_ids = sample_batch["sample_id"]

target_masses = batch_targets.flatten(start_dim=1).sum(dim=1)

print("Image shape:", tuple(batch_images.shape))
print("Target shape:", tuple(batch_targets.shape))
print("Image dtype:", batch_images.dtype)
print("Target dtype:", batch_targets.dtype)
print("First sample IDs:", list(batch_ids[:3]))
print("Target masses:", target_masses[:5])

assert batch_images.shape[1:] == (3, *OUTPUT_SIZE)
assert batch_targets.shape[1:] == (1, *OUTPUT_SIZE)
assert torch.allclose(
    target_masses,
    torch.ones_like(target_masses),
    atol=1e-4,
)

## 17. Visualize two preprocessed pairs

RGB normalization is reversed only for display. The saliency target remains a normalized spatial
distribution.

In [ ]:
import matplotlib.pyplot as plt

imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

figure, axes = plt.subplots(2, 2, figsize=(10, 7))

for row in range(2):
    rgb = (
        batch_images[row].cpu() * imagenet_std + imagenet_mean
    ).clamp(0.0, 1.0)

    axes[row, 0].imshow(rgb.permute(1, 2, 0))
    axes[row, 0].set_title(f"RGB — ID {batch_ids[row]}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(batch_targets[row, 0], cmap="magma")
    axes[row, 1].set_title("Normalized saliency target")
    axes[row, 1].axis("off")

figure.tight_layout()
plt.show()

# Part C — Heavy-M architecture checks

Heavy-M uses an ImageNet-pretrained ResNet-18 encoder and the **same** compact multi-scale decoder as
Light-M. This makes the Light-M versus Heavy-M comparison controlled: the principal architectural change is
the encoder capacity.

The four ResNet stages provide:

- `s4`: fine spatial detail;
- `s8`: intermediate local structure;
- `s16`: stronger semantic context;
- `s32`: the deepest global representation.


## 18. Build the Heavy-M model

The shared model factory creates ResNet-18 plus the common multi-scale decoder. Parameter counts are saved
separately for the encoder, decoder, complete model, and currently trainable parameters.


In [ ]:
model = build_model(
    MODEL_NAME,
    pretrained=PRETRAINED,
    decoder_channels=DECODER_CHANNELS,
).to(DEVICE)

model_parameters = parameter_summary(model)
parameter_dataframe = pd.DataFrame(
    [
        {
            "model": MODEL_NAME,
            **model_parameters,
        }
    ]
)
parameter_dataframe.to_csv(PARAMETER_SUMMARY_PATH, index=False)

display(parameter_dataframe)
print(model)

## 19. Verify the four ResNet-18 feature scales

For a `192 × 256` input, the selected ResNet-18 stages should produce the expected stride-4, stride-8,
stride-16, and stride-32 feature maps. The channel counts should be `64`, `128`, `256`, and `512`.


In [ ]:
model.eval()

with torch.inference_mode():
    feature_example = batch_images[:1].to(DEVICE)
    feature_maps = model.encoder(feature_example)

expected_channels = {
    "s4": 64,
    "s8": 128,
    "s16": 256,
    "s32": 512,
}

for feature_name, feature_tensor in feature_maps.items():
    print(f"{feature_name}: {tuple(feature_tensor.shape)}")

assert set(feature_maps) == set(expected_channels)

for feature_name, expected_channel_count in expected_channels.items():
    actual_channel_count = feature_maps[feature_name].shape[1]

    if actual_channel_count != expected_channel_count:
        raise RuntimeError(
            f"{feature_name} has {actual_channel_count} channels; "
            f"expected {expected_channel_count}."
        )


## 20. Run a forward, distribution, loss, and gradient test

This test catches output-shape errors, invalid spatial normalization, disconnected ResNet stages, disconnected
decoder stages, and numerical failures before the optimizer performs any update.


In [ ]:
model.train()
model.zero_grad(set_to_none=True)

sanity_images = batch_images[:2].to(DEVICE)
sanity_targets = batch_targets[:2].to(DEVICE)

sanity_logits = model(sanity_images)
sanity_predictions = spatial_softmax(sanity_logits)
sanity_loss_value = saliency_loss(sanity_logits, sanity_targets)

print("Logit shape:", tuple(sanity_logits.shape))
print(
    "Prediction masses:",
    sanity_predictions.flatten(start_dim=1).sum(dim=1).detach().cpu(),
)
print("Sanity loss:", float(sanity_loss_value.detach()))

assert sanity_logits.shape == sanity_targets.shape
assert torch.isfinite(sanity_loss_value)
assert torch.allclose(
    sanity_predictions.flatten(start_dim=1).sum(dim=1),
    torch.ones(sanity_images.shape[0], device=DEVICE),
    atol=1e-4,
)

sanity_loss_value.backward()

encoder_has_gradient = any(
    parameter.grad is not None
    for parameter in model.encoder.parameters()
    if parameter.requires_grad
)
decoder_has_gradient = any(
    parameter.grad is not None
    for parameter in model.decoder.parameters()
    if parameter.requires_grad
)

print("Encoder receives gradients:", encoder_has_gradient)
print("Decoder receives gradients:", decoder_has_gradient)

if not encoder_has_gradient or not decoder_has_gradient:
    raise RuntimeError("The forward/backward architecture check failed.")

model.zero_grad(set_to_none=True)

# Part D — Persistent training

The shared training loop saves history and `last.pt` after every epoch, writes `best.pt` whenever validation
CC improves, updates the required overfitting curves, and supports interruption-safe resume.

## 21. Build and save the run configuration

The same dictionary is stored as JSON and embedded inside every checkpoint.

In [ ]:
experiment_config = {
    "model_name": MODEL_NAME,
    "model_label": MODEL_LABEL,
    "git_commit": commit_hash,
    "temporary_source_patch_notes": source_patch_notes,
    "seed": SEED,
    "output_size": list(OUTPUT_SIZE),
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    "pretrained": PRETRAINED,
    "decoder_channels": DECODER_CHANNELS,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "minimum_cc_improvement": MIN_CC_IMPROVEMENT,
    "encoder_learning_rate": ENCODER_LR,
    "decoder_learning_rate": DECODER_LR,
    "weight_decay": WEIGHT_DECAY,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "freeze_encoder_epochs": FREEZE_ENCODER_EPOCHS,
    "use_amp_requested": USE_AMP,
    "use_amp_active": AMP_ENABLED,
    "checkpoint_metric": "validation_cc",
    "training_data_root": str(TRAINING_DATA_ROOT),
    "source_train_manifest": str(SOURCE_MANIFESTS["train"]),
    "source_val_manifest": str(SOURCE_MANIFESTS["val"]),
    "local_train_manifest": str(LOCAL_MANIFESTS["train"]),
    "local_val_manifest": str(LOCAL_MANIFESTS["val"]),
    "loader_benchmark": loader_benchmark,
    "parameter_summary": model_parameters,
}

with CONFIG_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=2)

print("Configuration saved to:", CONFIG_JSON_PATH)

## 22. Create the shared optimizer

The pretrained encoder uses a lower learning rate than the newly initialized decoder. The policy is
identical to the other neural models.

In [ ]:
optimizer = build_optimizer(
    model,
    encoder_lr=ENCODER_LR,
    decoder_lr=DECODER_LR,
    weight_decay=WEIGHT_DECAY,
)

for group in optimizer.param_groups:
    print(
        f"{group.get('name', 'unnamed')}: "
        f"lr={group['lr']:.2e}, "
        f"parameters={sum(parameter.numel() for parameter in group['params']):,}"
    )

## 23. Select resume behavior safely

With `RESUME_TRAINING=True`, an existing `last.pt` continues from the following epoch. Starting from
scratch while a previous run exists is blocked unless the explicit overwrite flag is enabled.

In [ ]:
resume_from = None

if LAST_CHECKPOINT_PATH.is_file() and RESUME_TRAINING:
    resume_from = LAST_CHECKPOINT_PATH
    print("Training will resume from:", resume_from)

elif LAST_CHECKPOINT_PATH.is_file() and not RESUME_TRAINING:
    if not ALLOW_OVERWRITE_EXISTING_RUN:
        raise RuntimeError(
            "An existing last.pt was found. Set RESUME_TRAINING=True to continue, "
            "or set ALLOW_OVERWRITE_EXISTING_RUN=True only when the old run should "
            "be replaced."
        )

    print("Existing run will be overwritten after training starts.")

else:
    print("No previous Heavy-M checkpoint found. Training will start at epoch 1.")

## 24. Freeze one validation preview batch

The same examples are shown after each epoch. This makes it easier to detect centre-collapse, constant
predictions, or incorrect spatial alignment.

In [ ]:
preview_batch = next(iter(val_loader))

print("Preview image shape:", tuple(preview_batch["image"].shape))
print("Preview sample IDs:", list(preview_batch["sample_id"][:4]))

## 25. Train and validate Heavy-M

This is the long-running cell. It can be executed again after a Colab interruption because `last.pt` restores
the model, optimizer, scaler, epoch history, and early-stopping state.

Heavy-M is larger than the two MobileNetV2 models, so its epoch time and GPU-memory use are expected to be
higher. The local SALICON cache prevents Drive file I/O from becoming the dominant bottleneck.


In [ ]:
history = fit_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    DEVICE,
    model_name=MODEL_NAME,
    epochs=MAX_EPOCHS,
    checkpoint_dir=MODEL_CHECKPOINT_DIR,
    history_csv_path=HISTORY_CSV_PATH,
    curves_figure_path=CURVES_FIGURE_PATH,
    preview_batch=preview_batch,
    preview_figure_path=PREVIEW_FIGURE_PATH,
    config=experiment_config,
    split_seed=SEED,
    repo_root=REPO_ROOT,
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=MIN_CC_IMPROVEMENT,
    freeze_encoder_epochs=FREEZE_ENCODER_EPOCHS,
    freeze_encoder_batchnorm=True,
    use_amp=USE_AMP,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    resume_from=resume_from,
    show_curves_each_epoch=True,
)

print(f"Stored history rows: {len(history)}")

## 26. Review the complete Heavy-M epoch history

The required plots contain training versus validation loss, validation KLD, and validation CC/SIM. They are
regenerated from the persistent CSV rather than relying on notebook memory.


In [ ]:
history_dataframe = pd.read_csv(HISTORY_CSV_PATH)

display(history_dataframe)

plot_training_history(
    history_dataframe.to_dict(orient="records"),
    output_path=CURVES_FIGURE_PATH,
    show=True,
    title="Heavy-M training history",
)

best_history_row = history_dataframe.loc[
    history_dataframe["val_cc"].idxmax()
]

print("Best validation epoch:", int(best_history_row["epoch"]))
print("Best validation CC:", float(best_history_row["val_cc"]))
print("Validation SIM at best epoch:", float(best_history_row["val_sim"]))
print("Validation KLD at best epoch:", float(best_history_row["val_kld"]))

# Part E — Load the best model and save validation results

The last in-memory weights may belong to a later epoch. The official training result is therefore loaded
from `best.pt`, selected only by validation CC.

## 27. Restore the best checkpoint

Optimizer state is not required for evaluation, so only the model is restored.

In [ ]:
best_checkpoint = load_checkpoint(
    BEST_CHECKPOINT_PATH,
    model=model,
    device=DEVICE,
    optimizer=None,
    scaler=None,
)

model.eval()

print("Loaded checkpoint epoch:", best_checkpoint.get("epoch"))
print("Best validation CC:", best_checkpoint.get("best_val_cc"))
print("Checkpoint Git commit:", best_checkpoint.get("git_commit"))

## 28. Evaluate the best checkpoint on the validation split

Dataset-level and per-image KLD, CC, and SIM values are saved. Per-image scores will later support median,
worst-case, qualitative, and centre-bias analyses.

In [ ]:
validation_summary, validation_per_image = evaluate_and_save(
    model,
    val_loader,
    DEVICE,
    model_name=MODEL_NAME,
    summary_csv_path=VALIDATION_SUMMARY_PATH,
    per_image_csv_path=VALIDATION_PER_IMAGE_PATH,
    use_amp=USE_AMP,
)

display(pd.DataFrame([validation_summary]))
display(validation_per_image.head())

## 29. Save a complete run summary

This compact JSON file records the environment, selected epoch, final validation metrics, model size,
persistent paths, and total epoch timing.

In [ ]:
total_training_seconds = float(history_dataframe["epoch_time_seconds"].sum())

run_summary = {
    "model": MODEL_NAME,
    "model_label": MODEL_LABEL,
    "status": "completed",
    "git_commit": commit_hash,
    "device": str(DEVICE),
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if DEVICE.type == "cuda"
        else None
    ),
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "best_checkpoint_epoch": int(best_checkpoint.get("epoch", 0)),
    "best_validation_cc": float(best_checkpoint.get("best_val_cc")),
    "validation_summary": validation_summary,
    "parameter_summary": model_parameters,
    "completed_epochs": int(history_dataframe["epoch"].max()),
    "total_epoch_time_seconds": total_training_seconds,
    "total_epoch_time_minutes": total_training_seconds / 60.0,
    "best_checkpoint_path": str(BEST_CHECKPOINT_PATH),
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
    "history_csv_path": str(HISTORY_CSV_PATH),
    "curves_figure_path": str(CURVES_FIGURE_PATH),
    "preview_figure_path": str(PREVIEW_FIGURE_PATH),
    "validation_summary_path": str(VALIDATION_SUMMARY_PATH),
    "validation_per_image_path": str(VALIDATION_PER_IMAGE_PATH),
    "config_path": str(CONFIG_JSON_PATH),
}

with RUN_SUMMARY_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2)

print("Run summary saved to:", RUN_SUMMARY_JSON_PATH)

## 30. Optional validation comparison with Light-M

When the Light-M history exists, this cell creates a small machine-readable table comparing the best
validation epochs of the two multi-scale models. This isolates encoder capacity under the shared decoder.

This validation table is useful for training diagnostics, but it does not replace the frozen final-test
comparison.


In [ ]:
LIGHT_MULTI_HISTORY_PATH = METRICS_ROOT / "light_multi_history.csv"
VALIDATION_COMPARISON_PATH = (
    METRICS_ROOT / "light_multi_vs_heavy_multi_validation.csv"
)

comparison_rows = []

for candidate_name, candidate_path in [
    ("light_multi", LIGHT_MULTI_HISTORY_PATH),
    ("heavy_multi", HISTORY_CSV_PATH),
]:
    if not candidate_path.is_file():
        print(f"Skipping missing history: {candidate_path}")
        continue

    candidate_history = pd.read_csv(candidate_path)
    candidate_best = candidate_history.loc[
        candidate_history["val_cc"].idxmax()
    ]

    comparison_rows.append(
        {
            "model": candidate_name,
            "best_epoch": int(candidate_best["epoch"]),
            "val_loss": float(candidate_best["val_loss"]),
            "val_kld": float(candidate_best["val_kld"]),
            "val_cc": float(candidate_best["val_cc"]),
            "val_sim": float(candidate_best["val_sim"]),
            "epoch_time_seconds": float(
                candidate_best.get("epoch_time_seconds", float("nan"))
            ),
        }
    )

validation_comparison = pd.DataFrame(comparison_rows)

if not validation_comparison.empty:
    validation_comparison.to_csv(
        VALIDATION_COMPARISON_PATH,
        index=False,
    )
    display(validation_comparison)
    print("Comparison saved to:", VALIDATION_COMPARISON_PATH)


## 31. Audit every required persistent artifact

The run is complete only when the checkpoints, numerical histories, figures, validation metrics, and
metadata files all exist in Drive.

In [ ]:
required_artifacts = [
    BEST_CHECKPOINT_PATH,
    LAST_CHECKPOINT_PATH,
    HISTORY_CSV_PATH,
    VALIDATION_SUMMARY_PATH,
    VALIDATION_PER_IMAGE_PATH,
    PARAMETER_SUMMARY_PATH,
    CURVES_FIGURE_PATH,
    PREVIEW_FIGURE_PATH,
    CONFIG_JSON_PATH,
    RUN_SUMMARY_JSON_PATH,
]

missing_artifacts = [
    path
    for path in required_artifacts
    if not path.is_file()
]

if missing_artifacts:
    raise RuntimeError(
        "The Heavy-M run is missing required artifacts:\n"
        + "\n".join(str(path) for path in missing_artifacts)
    )

artifact_table = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "path": str(path),
            "size_mb": path.stat().st_size / (1024 ** 2),
        }
        for path in required_artifacts
    ]
)

display(artifact_table)

print("All required Heavy-M artifacts are safely stored in Google Drive.")

# Heavy-M training complete

Heavy-M is the final neural training run in the controlled experiment.

The next notebook should perform the **frozen final evaluation**:

1. load `best.pt` for Light-S, Light-M, and Heavy-M;
2. evaluate all models on the same internal test split;
3. compare KLD, CC, and SIM against MeanMap;
4. measure parameters, model size, MACs/FLOPs, and CPU latency;
5. evaluate the off-centre subset;
6. generate aligned qualitative comparisons and final report tables.

No architecture, loss, split, or checkpoint-selection rule should be changed after this point.
